# 수혁님 — `B3_px1024`

**이번 실행에서 확인하는 것** — 사진 화질을 **640** 으로 올리면 512(C1, **0.5921**)보다 더 오르는지. 간판 글씨·가격을 읽는 문제가 많아서 화질이 현재 가장 효과 큰 변수입니다.

**예상 소요 시간** — 학습 약 1시간 20분 + dev 평가 약 25분

---

### 하는 법

1. **이전에 쓰던 노트북 커널을 먼저 종료하세요.** (이전 노트북 창 오른쪽 위 `Restart` 또는 창 닫기)
   이전 모델이 GPU 메모리를 잡고 있으면 이번 실행이 메모리 부족(OOM)으로 멈춥니다.
2. 이 노트북의 커널이 **baseline** 인지 확인 (오른쪽 위)
3. **모두 실행 (Run All)** — 설정은 이미 다 들어가 있어서 **아무것도 바꾸지 않아도 됩니다**
4. 끝나면 맨 아래 **11번 체크리스트**대로 업로드

### 중간에 눈으로 확인할 것 (세 군데)

| 셀 | 이렇게 나오면 정상 |
|---|---|
| 3번 데이터 로드 | `평가셋: dev 다수결(5표, 동률제외) 2216행` |
| 6번 마스킹 검증 | `loss 대상 디코딩: 'a<|im_end|>\n'` 처럼 **정답 한 글자 + im_end** |
| 10번 dev 평가 | `dev accuracy: 0.xxxx` → **이 숫자를 단톡에 공유** |

> 오늘 실행 중 **가장 빨리 시작해야 하는** 실행입니다. C1 업로드 끝나면 바로 시작해주세요.

### 에러가 나면

아무것도 고치지 말고 **에러 메시지 전체를 캡처**해서 동현에게 보내주세요.

In [ ]:
CONFIG = {
    "OWNER": "수혁",   # 담당자
    "VARIABLE": "IMAGE_SIZE",   # 이 노트북이 검증하는 변수
    "RUN_NAME": "B3_px1024",   # 실행마다 반드시 변경
    "N_TRAIN": 4000,   # 학습 샘플 수 (전량 6714)
    "IMAGE_SIZE": 1024,   # 이미지 한 변 픽셀
    "LORA_R": 16,   # LoRA rank (alpha 는 자동으로 2배)
    "BATCH_SIZE": 1,   # OOM 나면 1로
    "GRAD_ACCUM": 8,   # 실효 배치 = BATCH_SIZE x GRAD_ACCUM
    "EPOCHS": 1,
    "LR": 0.0001,
    "SEED": 42,   # 학습 시드
    "SPLIT_SEED": 42,   # 데이터 샘플링 시드 — 전원 동일하게 고정, 변경 금지
    "N_DEV": None,   # dev 평가 문항 수. None = 동률 제외 전체(권장)
    "MAX_HOURS": 7.0,   # 학습 상한(시간). = 18시 - 시작시각 - 1.5h. 9시 시작이면 7.0
}

# ↑ 이번 실행용으로 이미 설정 완료. 수정하지 마세요.
print(CONFIG["RUN_NAME"], "|", CONFIG["VARIABLE"])

# 1. 경로 · 오프라인 설정

팀 공통 폴더 구조입니다. 이 노트북은 `챌린지/` 바로 아래에 둡니다.

```
챌린지/
├─ baseline/                      ← 가상환경 (커널로 선택)
├─ data/
│  ├─ dev/ test/ train/           ← 이미지
│  └─ dev.csv test.csv train.csv sample_submission.csv
├─ downloads/
│  ├─ libs/                       ← wheel
│  └─ models/                     ← 사전학습 모델
├─ output/                        ← 자동 생성 (체크포인트 · 제출 · 실험기록)
└─ v4_?_*.ipynb                   ← 이 노트북
```

경로는 아래 셀에서 **자동 탐지**합니다. 노트북을 하위 폴더로 옮겨도 `data/` 와 `downloads/` 가
있는 상위 폴더를 찾아 올라가므로 수정할 필요가 없습니다.

In [ ]:
import os
from pathlib import Path

# --- 프로젝트 루트 자동 탐지 : data/ 와 downloads/ 가 함께 있는 폴더를 위로 올라가며 찾음 ---
def find_project_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "data").is_dir() and (cand / "downloads").is_dir():
            return cand
    raise FileNotFoundError(
        f"data/ 와 downloads/ 가 있는 폴더를 찾지 못했습니다. 현재 위치: {p}\n"
        "노트북을 '챌린지' 폴더 안에 두고 다시 실행하세요."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR     = PROJECT_ROOT / "data"
ASSET_DIR    = PROJECT_ROOT / "downloads"
LIB_DIR      = ASSET_DIR / "libs"
OUTPUT_DIR   = PROJECT_ROOT / "output"
CKPT_DIR     = OUTPUT_DIR / "ckpt"
SUB_DIR      = OUTPUT_DIR / "submissions"
for d in (OUTPUT_DIR, CKPT_DIR, SUB_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- 모델 폴더 자동 탐지 : downloads/models 아래 이름이 무엇이든 찾아냄 ---
def find_model_dir(models_root: Path):
    if not models_root.is_dir():
        raise FileNotFoundError(f"{models_root} 가 없습니다. 사전 다운로드 노트북을 먼저 실행하세요.")
    cands = [
        cfg.parent for cfg in sorted(models_root.rglob("config.json"))
        if (cfg.parent / "preprocessor_config.json").exists()
    ]
    if not cands:
        raise FileNotFoundError(
            f"{models_root} 안에서 모델을 찾지 못했습니다. "
            "(config.json + preprocessor_config.json 이 같이 있는 폴더가 필요)"
        )
    qwen = [c for c in cands if "qwen" in c.as_posix().lower()]
    return (qwen or cands)[0]

MODEL_DIR = find_model_dir(ASSET_DIR / "models")

# HuggingFace 오프라인 모드 : 모델 로드 시 인터넷 접속 시도 자체를 차단
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("PROJECT_ROOT :", PROJECT_ROOT)
for name, path in [("라이브러리", LIB_DIR), ("모델", MODEL_DIR),
                   ("데이터", DATA_DIR), ("출력", OUTPUT_DIR)]:
    ok = path.is_dir() and any(path.iterdir())
    print(f"  {name:<6s} {'OK' if ok else '없음 -> 사전 준비 필요':<18s} {path}")

for f in ["train.csv", "dev.csv", "test.csv", "sample_submission.csv"]:
    print(f"  {f:<22s} {'OK' if (DATA_DIR / f).exists() else '없음'}")

# 2. 환경 준비 (RTX 5060 Ti / Blackwell sm_120)

미리 받아둔 `downloads/libs` wheel 로 설치합니다. **인터넷에 접속하지 않습니다.**

> **Blackwell 주의** — 5060 Ti 는 sm_120 이라 아래 3가지가 베이스라인과 다릅니다.
> 1. `torch` 는 반드시 **cu128** 빌드 (cu121 빌드는 동작하지 않음 → 콜랩 노트북 설치 셀 복붙 금지)
> 2. `bitsandbytes` 는 sm_120 커널이 포함된 **최신 버전** 필요 (0.43.x 에서는 4bit 로드 실패 가능)
> 3. 모델 로드 시 `attn_implementation="eager"` 필수 (flash-attn / Triton 커널 미지원)

In [ ]:
import sys
# 미리 다운로드한 wheel 로 설치 (인터넷 미사용). CUDA 12.8 빌드로 고정
# %pip 대신 {sys.executable} 을 쓰는 이유 : pip 와 커널의 경로가 어긋나 엉뚱한 환경에 설치되는 것을 방지
!{sys.executable} -m pip install -q --no-index --find-links "{LIB_DIR}" torch torchvision torchaudio

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
assert "+cu" in torch.__version__, (
    f"CPU 전용 torch({torch.__version__}) 가 설치되어 있습니다. "
    "downloads/libs 에 torch-*+cu128 wheel 이 있는지 확인하고 위 셀을 다시 실행하세요."
)
print("device:", torch.cuda.get_device_name())
cap = torch.cuda.get_device_capability()
print("compute capability:", cap)          # 5060 Ti -> (12, 0)
print("VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
!{sys.executable} -m pip install -q --no-index --find-links "{LIB_DIR}" "transformers>=4.49.0,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.45.0" datasets pillow pandas --upgrade

# bitsandbytes 가 sm_120 을 인식하는지 확인 (여기서 에러가 나면 4bit 학습 자체가 불가)
import bitsandbytes as bnb
print("bitsandbytes:", bnb.__version__)

# 3. 데이터 로드

- `train.csv` 전체 6,714행 중 `CONFIG["N_TRAIN"]` 개만 사용합니다.
- **평가셋** : `dev.csv` 는 문항당 5명이 답한 구조(`answer1~5`)라 **다수결을 정답**으로 쓰고, 1위 동률 문항은 제외합니다.
  - **dev 라벨은 의도적으로 노이즈가 섞인 구조입니다.** 5표 중 한 보기가 4표 이상 받은 문항이 하나도 없고
    (최대 3표), 만장일치는 0.8% 입니다. 다수결은 최선의 추정일 뿐 정답 보장이 아니므로
    **dev_acc 의 절대값은 실제 test 점수보다 낮게 나올 수 있습니다.** 다만 4명이 같은 규칙으로 재므로
    실험 간 **상대 비교**에는 그대로 유효합니다.
  - 노이즈 때문에 표본이 적으면 흔들림이 커서, 기본값은 **동률 제외 dev 전체(약 2,216문항)** 로 평가합니다.
  - dev 는 라벨 노이즈 때문에 **학습 데이터로 쓰지 않습니다.**
  (dev 에 정답이 전혀 없는 경우를 대비해 **train 고정 홀드아웃 300개**로 폴백하는 로직도 남겨뒀습니다.)
  홀드아웃은 `SPLIT_SEED` 순열의 마지막 300개라, `N_TRAIN` 을 몇으로 하든 학습셋에 절대 포함되지 않습니다.

- `train.csv` 의 `path` 컬럼이 어느 폴더 기준인지는 첫 행으로 **실측해서 자동 결정**합니다.
- 샘플링 시드는 `SPLIT_SEED` 로 **전원 동일하게 고정**합니다. 이게 달라지면 서로 다른 데이터로 학습하게 되어 비교가 깨집니다.

In [ ]:
import re, math, random, time, json
import numpy as np
import pandas as pd
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

SEED = CONFIG["SEED"]
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

train_df_full = pd.read_csv(DATA_DIR / "train.csv")
dev_df        = pd.read_csv(DATA_DIR / "dev.csv")
test_df       = pd.read_csv(DATA_DIR / "test.csv")
sample_sub    = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train.csv 컬럼:", list(train_df_full.columns))
print("sample_submission.csv 컬럼:", list(sample_sub.columns))

# --- 이미지 경로 기준 자동 탐지 : path 컬럼이 'train/x.jpg' 인지 'data/train/x.jpg' 인지 모르므로 실측 ---
def resolve_image_base(df):
    rel = str(df["path"].iloc[0]).replace("\\", "/")
    for base in [DATA_DIR, PROJECT_ROOT]:
        if (base / rel).exists():
            return base
    # path 가 파일명만 있는 경우 (train/ 폴더 직속)
    for sub in ["train", "dev", "test"]:
        if (DATA_DIR / sub / rel).exists():
            return DATA_DIR / sub
    raise FileNotFoundError(f"이미지를 찾지 못했습니다: path='{rel}' (DATA_DIR={DATA_DIR})")

IMG_BASE = resolve_image_base(train_df_full)
print("이미지 기준 경로:", IMG_BASE)

def image_path(rel):
    return IMG_BASE / str(rel).replace("\\", "/")

assert image_path(train_df_full["path"].iloc[0]).exists()
assert image_path(dev_df["path"].iloc[0]).exists()
assert image_path(test_df["path"].iloc[0]).exists()
print("train/dev/test 이미지 경로 확인 완료")

# --- 평가셋 결정 : dev.csv 에 정답 컬럼이 있으면 dev, 없으면 train 홀드아웃 ---
HOLDOUT_N = 300
perm = np.random.RandomState(CONFIG["SPLIT_SEED"]).permutation(len(train_df_full))

print("dev.csv 컬럼:", list(dev_df.columns))
from collections import Counter
_LET = ["a", "b", "c", "d"]

def to_letter(v, row):
    # 투표값을 a/b/c/d 로 정규화. 빈 투표(NaN)는 무시, 보기 텍스트로 적혀 있으면 a~d 컬럼과 대조
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
    if s in _LET:
        return s
    for L in _LET:
        if str(row[L]).strip().lower() == s:
            return L
    return None

def majority(votes):
    # 최다 득표 보기. 1위 동률이면 None (평가에서 제외)
    top = Counter(v for v in votes if v).most_common()
    if not top or (len(top) > 1 and top[0][1] == top[1][1]):
        return None
    return top[0][0]

VOTE_COLS = sorted(c for c in dev_df.columns if re.fullmatch(r"answer\d+", c))
LABEL_CANDS = ["answer", "label", "gold", "target", "정답"]
dev_label = next((c for c in LABEL_CANDS if c in dev_df.columns), None)

if VOTE_COLS:
    # dev 는 문항당 여러 명이 답한 구조 -> 다수결을 정답으로, 1위 동률 문항은 제외
    pool_idx = perm                                   # 전량 학습 가능
    d = dev_df.copy()
    d["votes"]  = d.apply(lambda r: [to_letter(r[c], r) for c in VOTE_COLS], axis=1)
    d["answer"] = d["votes"].apply(majority)
    n_tie = int(d["answer"].isna().sum())
    d = d[d["answer"].notna()]
    n_eval = min(CONFIG["N_DEV"] or len(d), len(d))
    dev_eval_df = d.sample(n=n_eval, random_state=CONFIG["SPLIT_SEED"]).reset_index(drop=True)
    unanimous = dev_eval_df["votes"].apply(lambda v: len({x for x in v if x}) == 1).mean()
    print(f"투표 컬럼 {VOTE_COLS} / dev {len(dev_df)}행 중 동률 {n_tie}행 제외")
    print(f"평가셋 만장일치 비율: {unanimous:.3f}")
    EVAL_SOURCE = f"dev 다수결({len(VOTE_COLS)}표, 동률제외)"
elif dev_label is not None:
    pool_idx = perm                                   # 전량 학습 가능
    dev_eval_df = dev_df.rename(columns={dev_label: "answer"})
    if CONFIG["N_DEV"] and CONFIG["N_DEV"] < len(dev_eval_df):
        dev_eval_df = dev_eval_df.sample(n=CONFIG["N_DEV"], random_state=CONFIG["SPLIT_SEED"])
    dev_eval_df = dev_eval_df.reset_index(drop=True)
    EVAL_SOURCE = f"dev.csv ('{dev_label}')"
else:
    # 순열 마지막 300개 = 어떤 N_TRAIN 으로 학습해도 절대 포함되지 않는 행
    pool_idx = perm[:-HOLDOUT_N]
    dev_eval_df = train_df_full.iloc[perm[-HOLDOUT_N:]].reset_index(drop=True)
    EVAL_SOURCE = f"train 홀드아웃 {HOLDOUT_N}"

# --- 학습 데이터 : 순열 앞에서 N_TRAIN 개 (df.sample(n, random_state) 와 동일한 결과) ---
n = CONFIG["N_TRAIN"]
if n > len(pool_idx):
    print(f"[알림] N_TRAIN {n} > 학습 풀 {len(pool_idx)} -> {len(pool_idx)} 로 제한")
    n = len(pool_idx)
train_df = train_df_full.iloc[pool_idx[:n]].reset_index(drop=True)

overlap = set(dev_eval_df["path"]) & set(train_df["path"])
assert not overlap, f"평가셋이 학습셋과 {len(overlap)}개 겹칩니다"
print(f"평가셋: {EVAL_SOURCE} {len(dev_eval_df)}행 (학습셋과 겹침 0)")

print(f"train {len(train_df)}행 / dev(평가) {len(dev_eval_df)}행 / test {len(test_df)}행")
print("정답 분포(train):", train_df["answer"].astype(str).str.strip().str.lower().value_counts().to_dict())

# 4. 모델 · Processor · LoRA

베이스라인 대비 바뀐 점:

| 항목 | 베이스라인 | 이 노트북 | 이유 |
|---|---|---|---|
| `attn_implementation` | 없음 | `"eager"` | sm_120 에서 flash/Triton 커널 미지원 |
| `bnb_4bit_compute_dtype` | `float16` | `bfloat16` | autocast 가 bf16 이라 dtype 을 맞춤 |
| `IMAGE_SIZE` | 384 고정 | `CONFIG["IMAGE_SIZE"]` | 해상도를 실험 변수로 사용 |
| LoRA `r` | 8 고정 | `CONFIG["LORA_R"]` | LoRA 용량을 실험 변수로 사용 |

In [ ]:
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

IMAGE_SIZE = CONFIG["IMAGE_SIZE"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # autocast(bf16) 과 일치시킴
)

processor = AutoProcessor.from_pretrained(
    MODEL_DIR,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
    local_files_only=True,
)
# 학습 시에는 오른쪽 패딩이어야 라벨 마스킹 위치 계산이 맞습니다
processor.tokenizer.padding_side = "right"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True,
    attn_implementation="eager",     # Blackwell 필수
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base_model.config.use_cache = False

lora_config = LoraConfig(
    r=CONFIG["LORA_R"],
    lora_alpha=CONFIG["LORA_R"] * 2,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 5. 프롬프트 템플릿

베이스라인과 **동일하게 유지**합니다. 프롬프트를 바꾸면 4명의 결과가 서로 비교 불가능해지므로,
이 셀은 누구도 수정하지 않습니다.

In [ ]:
SYSTEM_INSTRUCT = (
    "You are an expert Visual Question Answering assistant with superior OCR capability. "
    "Examine the image carefully, paying close attention to texts, numbers, signs, and small visual details. "
    "Select the best option and respond with EXACTLY ONE lowercase letter among a, b, c, or d. Do not output anything else."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"Question: {question}\n"
        f"Options:\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "Carefully read all texts and visual features in the image. "
        "Output ONLY the single lowercase letter (a, b, c, or d) corresponding to the correct answer."
    )

LETTERS = ["a", "b", "c", "d"]
LETTER_IDS = []
for ch in LETTERS:
    ids = processor.tokenizer.encode(ch, add_special_tokens=False)
    assert len(ids) == 1, f"'{ch}' 가 단일 토큰이 아닙니다: {ids}"
    LETTER_IDS.append(ids[0])
print("letter token ids:", dict(zip(LETTERS, LETTER_IDS)))

# 6. Dataset · Collator — **라벨 마스킹** (핵심 수정)

베이스라인의 치명적 버그:

```python
enc["labels"] = enc["input_ids"].clone()    # 프롬프트 전체에 loss 가 걸림
```

시스템 프롬프트 · 질문 · 선택지 4개 · 이미지 토큰(384px 기준 약 190개) 전부에 loss 가 걸립니다.
정작 학습시켜야 할 정답은 **토큰 1개**인데, 전체 수백 토큰에 희석되어 학습 신호가 거의 사라집니다.

아래 구현은 **정답 토큰 외 전부를 `-100`** 으로 막습니다.
- 프롬프트 구간: `add_generation_prompt=True` 로 프롬프트만 따로 인코딩해 길이를 구한 뒤 그 앞까지 마스킹
- 이미지 토큰: 프롬프트 구간에 포함되므로 같이 마스킹됨
- 패딩: `attention_mask == 0` 위치를 마스킹

In [ ]:
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any

class VQAMCDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(image_path(row["path"])).convert("RGB")
        user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user",   "content": [{"type": "image", "image": img},
                                           {"type": "text",  "text": user_text}]},
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role": "assistant", "content": [{"type": "text", "text": gold}]})

        return {"messages": messages, "image": img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        images      = [s["image"] for s in batch]
        full_texts  = []
        prompt_texts = []

        for s in batch:
            msgs = s["messages"]
            if self.train:
                full_texts.append(self.processor.apply_chat_template(
                    msgs, tokenize=False, add_generation_prompt=False))
                # 정답(assistant turn)을 뺀 프롬프트만 -> 마스킹 경계 계산용
                prompt_texts.append(self.processor.apply_chat_template(
                    msgs[:-1], tokenize=False, add_generation_prompt=True))
            else:
                full_texts.append(self.processor.apply_chat_template(
                    msgs, tokenize=False, add_generation_prompt=True))

        enc = self.processor(text=full_texts, images=images, padding=True, return_tensors="pt")

        if self.train:
            p_enc = self.processor(text=prompt_texts, images=images, padding=True, return_tensors="pt")
            prompt_lens = p_enc["attention_mask"].sum(dim=1)    # 패딩 제외 실제 프롬프트 길이

            labels = enc["input_ids"].clone()
            labels[enc["attention_mask"] == 0] = -100           # 패딩 마스킹
            for i, L in enumerate(prompt_lens):
                labels[i, :int(L)] = -100                       # 프롬프트 + 이미지 토큰 마스킹
            enc["labels"] = labels

        return enc


# --- 마스킹 검증 (반드시 확인) ---
_ds = VQAMCDataset(train_df.head(2), train=True)
_batch = DataCollator(processor, True)([_ds[0], _ds[1]])
_lab = _batch["labels"][0]
_kept = _lab[_lab != -100]
print("전체 토큰:", _batch["labels"].shape[1], "/ loss 대상 토큰:", len(_kept))
print("loss 대상 디코딩:", repr(processor.tokenizer.decode(_kept)))
print("=> 정답 한 글자 + 종료토큰 정도만 남아야 정상입니다")

# 7. DataLoader

In [ ]:
train_ds = VQAMCDataset(train_df, train=True)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=True,
    collate_fn=DataCollator(processor, True),
    num_workers=0,
)
print(f"train batches: {len(train_loader)} (batch_size={CONFIG['BATCH_SIZE']})")

# 8. dev 정확도 — **로짓 스코어링** (핵심 수정)

베이스라인은 `generate()` 로 글자를 생성한 뒤 텍스트를 파싱했고, 파싱에 실패하면 조용히 `"a"` 를 반환했습니다.
전체 test 추론에 30분~1시간이 걸린 것도 이 때문입니다.

여기서는 **forward 1회로 다음 토큰 로짓을 뽑아 a/b/c/d 4개만 비교**합니다.
- 항상 유효한 답이 나옴 (파싱 실패로 인한 `"a"` 편향 없음)
- 생성 루프가 없어 추론이 수 분대로 단축
- 4개 로짓에서 확률까지 뽑아두므로 나중에 **앙상블(확률 평균)** 에 그대로 재사용 가능

In [ ]:
@torch.no_grad()
def predict_letters(df, desc="predict", return_probs=False):
    """a/b/c/d 다음토큰 로짓을 비교해 예측. (preds, probs) 반환"""
    from tqdm.auto import tqdm
    model.eval()
    preds, probs = [], []

    for i in tqdm(range(len(df)), desc=desc, unit="it"):
        row = df.iloc[i]
        img = Image.open(image_path(row["path"])).convert("RGB")
        user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user",   "content": [{"type": "image", "image": img},
                                           {"type": "text",  "text": user_text}]},
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(**inputs).logits[:, -1, :].float()

        sel = logits[0, LETTER_IDS]
        preds.append(LETTERS[int(sel.argmax())])
        probs.append(torch.softmax(sel, dim=-1).cpu().tolist())

    model.train()
    return (preds, probs) if return_probs else (preds, None)


def dev_accuracy(df, desc="dev"):
    preds, probs = predict_letters(df, desc=desc, return_probs=True)
    gold = df["answer"].astype(str).str.strip().str.lower().tolist()
    acc = sum(p == g for p, g in zip(preds, gold)) / len(gold)
    from collections import Counter
    print(f"dev accuracy: {acc:.4f}  ({sum(p == g for p, g in zip(preds, gold))}/{len(gold)})")
    print("예측 분포:", dict(Counter(preds)), "/ 정답 분포:", dict(Counter(gold)))
    return acc, preds, probs

# 9. 학습

베이스라인 대비 바뀐 점:

- **`GradScaler` 제거** — GradScaler 는 fp16 전용입니다. 베이스라인은 autocast 를 bf16 으로 쓰면서
  scaler 를 같이 썼는데, bf16 에서는 불필요하고 동작도 어긋납니다.
- `torch.cuda.amp` (deprecated) → `torch.amp`
- **시간 상한(`MAX_HOURS`)** — 상한에 걸리면 루프를 멈추고 체크포인트를 저장합니다.
  5060 Ti 사용 가능 시간은 **09:00~18:00** 입니다. 학습 뒤에 dev 추론 · test 추론 · 업로드까지 끝나야 하므로
  기본값은 7.0시간입니다. **늦게 시작했으면 `18시 - 시작시각 - 1.5h` 로 줄이세요.**

In [ ]:
from tqdm.auto import tqdm

model.train()
GRAD_ACCUM = CONFIG["GRAD_ACCUM"]
EPOCHS     = CONFIG["EPOCHS"]

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CONFIG["LR"])
num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(num_training_steps * 0.03), num_training_steps
)
print(f"총 옵티마이저 스텝: {num_training_steps}")

deadline = time.time() + CONFIG["MAX_HOURS"] * 3600
train_t0 = time.time()
stopped_early = False

for epoch in range(EPOCHS):
    running, seen = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch")
    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            loss = model(**batch).loss / GRAD_ACCUM
        loss.backward()
        running += loss.item() * GRAD_ACCUM
        seen += 1

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            pbar.set_postfix({"loss": f"{running/seen:.4f}"})
            running, seen = 0.0, 0

        if time.time() > deadline:
            print(f"\n[시간 상한 {CONFIG['MAX_HOURS']}h 도달] epoch {epoch+1} step {step} 에서 중단하고 저장합니다.")
            stopped_early = True
            break
    if stopped_early:
        break

train_minutes = (time.time() - train_t0) / 60
print(f"학습 시간: {train_minutes:.1f}분 (early stop: {stopped_early})")

SAVE_DIR = CKPT_DIR / CONFIG["RUN_NAME"]
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)

# 10. dev 평가 · 실험 기록

결과는 `output/experiments.csv` 에 한 줄씩 쌓입니다. **이 파일을 팀 공용 시트에 그대로 옮겨 붙이세요.**
6일차에 무엇을 앙상블할지는 전적으로 이 표를 보고 정합니다.

In [ ]:
eval_t0 = time.time()
acc, dev_preds, dev_probs = dev_accuracy(dev_eval_df, desc=f"dev[{CONFIG['RUN_NAME']}]")
eval_minutes = (time.time() - eval_t0) / 60
print(f"dev 추론 시간: {eval_minutes:.1f}분")

# dev 확률 저장 (앙상블용)
np.save(CKPT_DIR / CONFIG["RUN_NAME"] / "dev_probs.npy", np.array(dev_probs))

row = {
    "run_name":    CONFIG["RUN_NAME"],
    "담당":         CONFIG["OWNER"],
    "변수":         CONFIG["VARIABLE"],
    "n_train":     len(train_df),
    "image_size":  CONFIG["IMAGE_SIZE"],
    "lora_r":      CONFIG["LORA_R"],
    "batch_size":  CONFIG["BATCH_SIZE"],
    "epochs":      CONFIG["EPOCHS"],
    "lr":          CONFIG["LR"],
    "seed":        CONFIG["SEED"],
    "dev_acc":     round(acc, 4),
    "n_dev":       len(dev_eval_df),
    "평가셋":       EVAL_SOURCE,
    "학습분":       round(train_minutes, 1),
    "추론분":       round(eval_minutes, 1),
    "early_stop":  stopped_early,
    "기록시각":     time.strftime("%Y-%m-%d %H:%M"),
}

log_path = OUTPUT_DIR / "experiments.csv"
log_df = pd.DataFrame([row])
if log_path.exists():
    log_df = pd.concat([pd.read_csv(log_path), log_df], ignore_index=True)
log_df.to_csv(log_path, index=False, encoding="utf-8-sig")

print("\n=== 누적 실험 기록 ===")
print(log_df[[c for c in ["run_name", "변수", "n_train", "image_size", "lora_r", "dev_acc", "학습분"] if c in log_df.columns]].to_string(index=False))

# 11. 끝나면 체크리스트

- [ ] 10번 셀의 `dev accuracy` 숫자를 단톡에 공유
- [ ] 공유 드라이브에 업로드
  - `output/experiments.csv`
  - `output/ckpt/C2_px640/` **폴더 통째로**
- [ ] 다 올렸으면 이 노트북 커널 종료 (다음 실행을 위해 GPU 메모리 비우기)

> 이번 실행에서는 **test 추론(제출 파일 생성)을 하지 않습니다.** 모델은 위 폴더에 저장되어 있어서, 필요하면 나중에 불러와 추론할 수 있습니다.